### Cohere Embedding

In [5]:
import cohere
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv(), override=True)

cohere_api_key = os.getenv("COHERE_API_KEY")
co = cohere.ClientV2(api_key=cohere_api_key)

source_texts = [
    "You're a wizard, Harry.",
    "Space, the final frontier.",
    "I'm going to make him an offer he can't refuse.",
]

response = co.embed(
    texts=source_texts,
    model="embed-english-light-v3.0",
    input_type="search_document",
    embedding_types=["float"],
)

source_embeddings = []
for e in response.embeddings.float_:
    print(len(e))                   # This will be the length of the embedding vector
    print(e[:5])                    # This will print the first 5 elements of the embedding vector
    source_embeddings.append(e)     # Save the embedding for later use

384
[0.024459839, 0.039001465, -0.013053894, 0.016342163, -0.049926758]
384
[-0.0051002502, 0.017578125, -0.0256958, 0.023513794, 0.018493652]
384
[-0.076660156, 0.04244995, -0.07366943, 0.0019054413, -0.007736206]


In [6]:
# Get the query embedding:
query_text = "Intergalactic voyage"

response = co.embed(
    texts=[query_text],
    model="embed-english-light-v3.0",
    input_type="search_query",
    embedding_types=["float"],
)

query_embedding = response.embeddings.float_[0]

print(len(query_embedding))
print(query_embedding[:5])

384
[-0.007019043, -0.097839355, 0.023117065, 0.0049324036, 0.047027588]


In [7]:
# Find the most similar source text to the query:
import numpy as np

# Calculate the dot product between the query embedding and each source embedding
dot_products = [np.dot(query_embedding, e) for e in source_embeddings]

# Find the index of the maximum dot product
most_similar_index = np.argmax(dot_products)

# Get the most similar source text
most_similar_text = source_texts[most_similar_index]

print(f"The most similar text to '{query_text}' is:")
print(most_similar_text)

The most similar text to 'Intergalactic voyage' is:
Space, the final frontier.


In [13]:
import cohere

co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))
response = co.chat(
    model="command-a-03-2025", 
    messages=[{"role": "user", "content": "hello world!"}]
)

print(response)


id='9b0b09a8-7b04-47e1-98fc-5c6fc2b50759' finish_reason='COMPLETE' message=AssistantMessageResponse(role='assistant', tool_calls=None, tool_plan=None, content=[TextAssistantMessageResponseContentItem(type='text', text='Hello! How can I assist you today?')], citations=None) usage=Usage(billed_units=UsageBilledUnits(input_tokens=3.0, output_tokens=9.0, search_units=None, classifications=None), tokens=UsageTokens(input_tokens=498.0, output_tokens=11.0), cached_tokens=0.0) logprobs=None


In [20]:
print(response.message.content[0].text)

Hello! How can I assist you today?


In [21]:
import cohere
import os

co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

response = co.chat(
    model="command-a-03-2025",   # ✅ safest fallback
    messages=[
        {
            "role": "user",
            "content": "Explain language models in 4 sentences."
        }
    ]
)

print(response.message.content[0].text)

Language models are computational tools designed to understand and generate human-like text by predicting the next word in a sequence based on patterns learned from large datasets. They use machine learning techniques, particularly neural networks, to capture the structure and nuances of language. These models can perform tasks such as translation, summarization, and conversation by leveraging their ability to process and generate coherent text. Examples include GPT, BERT, and T5, which have revolutionized natural language processing with their versatility and accuracy.


In [24]:
import cohere
import os

cohere_api_key = os.getenv("COHERE_API_KEY")
co = cohere.ClientV2(api_key=cohere_api_key)

messages = [
    {
        "role": "user",
        "content": "Hi there. Please explain how language models work, in just a 3 sentence or two.",
    }
]

# Initial response from the model
response = co.chat(
    model="command-a-03-2025",
    messages=messages,
)
print(response.message.content[0].text)

# Append the initial response to the messages
messages.append(
    {
        "role": "assistant",
        "content": response.message.content[0].text,
    }
)

# Provide a follow-up prompt
messages.append(
    {
        "role": "user",
        "content": "Ah, I see. Now, can you write that in a Haiku?",
    }
)

response = co.chat(
    model="command-a-03-2025",
    messages=messages,
)

# This response will take both the initial and follow-up prompts into account
print(response.message.content[0].text)

Language models, like me, are trained on vast amounts of text data to learn patterns, grammar, and context. They use this knowledge to predict the next word in a sequence, enabling them to generate coherent and contextually relevant responses to prompts or questions. Essentially, they work by statistically modeling the relationships between words and phrases to produce human-like text based on the input they receive.
Words, patterns, and thought,
Statistical dance of text,
Language models speak.


In [25]:
import ollama

source_texts = [
    "You're a wizard, Harry.",
    "Space, the final frontier.",
    "I'm going to make him an offer he can't refuse.",
]

response = ollama.embed(model='snowflake-arctic-embed:110m', input=source_texts)

source_embeddings = []
for e in response.embeddings:
    print(len(e))                   # This will be the length of the embedding vector
    print(e[:5])                    # This will print the first 5 elements of the embedding vector
    source_embeddings.append(e)     # Save the embedding for later use

768
[-0.030616712, 0.017595656, -0.0011778469, 0.025150131, 0.005873111]
768
[-0.03988855, 0.05197487, 0.036461893, 0.012905286, 0.0120680025]
768
[-0.049428858, 0.054666363, -0.007880886, -0.002527469, -0.0025287406]


In [26]:
# Get the query embedding:
query_text = "Intergalactic voyage"

response = ollama.embed(model='snowflake-arctic-embed:110m', input=query_text)

query_embedding = response.embeddings[0]

print(len(query_embedding))
print(query_embedding[:5])

768
[-0.04345638, 0.052611303, 0.025881145, -0.017231025, 0.0274321]


In [27]:
# Find the most similar source text to the query:
import numpy as np

# Calculate the dot product between the query embedding and each source embedding
dot_products = [np.dot(query_embedding, e) for e in source_embeddings]

# Find the index of the maximum dot product
most_similar_index = np.argmax(dot_products)

# Get the most similar source text
most_similar_text = source_texts[most_similar_index]


print(f"The most similar text to '{query_text}' is:")
print(most_similar_text)

The most similar text to 'Intergalactic voyage' is:
Space, the final frontier.


In [28]:
from ollama import chat
from ollama import ChatResponse

messages = [
    {
        "role": "user",
        "content": "Hi there. Please explain how language models work, in just a sentence or two.",
    }
]

response: ChatResponse = chat(model='llama3:latest', messages=messages)

print(response.message.content)

Language models are artificial intelligence algorithms that analyze vast amounts of text data to learn patterns and relationships between words, phrases, and ideas, allowing them to generate coherent and context-specific text based on input prompts or questions. They achieve this by processing massive datasets through neural networks, which enable the model to predict the next word in a sequence given the preceding words.


In [30]:
from ollama import chat
from ollama import ChatResponse

messages = [
    {
        "role": "user",
        "content": "Hi there. Please explain how language models work, in just 3 sentences or two.",
    }
]

# Initial response from the model
response: ChatResponse = chat(model='llama3:latest', messages=messages)
print(response.message.content)
print()
print()

# Append the initial response to the messages
messages.append(
    {
        "role": "assistant",
        "content": response.message.content,
    }
)

# Provide a follow-up prompt
messages.append(
    {
        "role": "user",
        "content": "Ah, I see. Now, can you write that in a Haiku?",
    }
)

response: ChatResponse = chat(model='llama3:latest', messages=messages)


# This response will take both the initial and follow-up prompts into account
print(response.message.content)

Here's a brief explanation:

Language models are artificial intelligence algorithms that analyze vast amounts of text data to learn patterns and relationships between words, allowing them to generate coherent and meaningful text. They do this by processing input text one word at a time, using a combination of natural language processing (NLP) techniques and machine learning algorithms to predict the next most likely word based on the context. The model then uses these predictions to generate new text that is similar in style and tone to the original training data.


Here's a Haiku:

Text patterns learned deep
Predicting next words with ease
Language flows anew
